In [1]:
import numpy as np
import pandas as pd
from scipy.stats import norm

In [ ]:
# Excercise 7.7
# Simulating the gain/loss from a delta-hedged portfolio at various percentiles 
def simulated_delta_hedge_profit(S0, K, r, sigma, q, T, mu, M, N, pct):
    """
    S0 = initial stock price
    K = strike price
    r = risk-free rate
    sigma = volatility
    q = dividend yield
    T = time to maturity
    mu = expected rate of return
    M = number of simulations
    N = number of time periods
    pct = list of percentiles to be returned
    """

    dt = T / N
    sigSqrdt = sigma * np.sqrt(dt)
    drift = (mu - q - 0.5*sigma**2) * dt
    compound = np.exp(r * dt)
    div_factor = np.exp(q * dt) - 1

    logS0 = np.log(S0)

    # Initial call price and delta
    d1 = (np.log(S0/K) + (r - q + 0.5*sigma**2)*T) / (sigma*np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)
    
    call0 = np.exp(-q*T) * S0 * norm.cdf(d1) - np.exp(-r*T) * K * norm.cdf(d2)
    delta0 = np.exp(-q*T) * norm.cdf(d1)
    cash0 = call0 - delta0 * S0

    # Array to store simulations
    profits = np.zeros(M)

    for i in range(M):
        logS = logS0
        cash = cash0
        S = S0
        delta = delta0

        for j in range(1, N):
            logS += drift + sigSqrdt * np.random.randn()
            newS = np.exp(logS)

            # Calculate new delta
            time_step = T - j*dt
            new_d1 = (np.log(newS/K) + (r - q + 0.5*sigma**2)*(time_step)) / (sigma*np.sqrt(time_step))
            new_delta = np.exp(-q*time_step) * norm.cdf(new_d1)

            cash = cash * compound + (delta - new_delta) * newS - div_factor * delta * S

            S = newS
            delta = new_delta

        final_call_price = max(0, S - K)
        profits[i] = cash + delta * S - final_call_price
    
    # Putting percentiles in a dictionary
    results = {p: np.percentile(profits, p) for p in pct}

    return results

In [ ]:
# Set parameters
S0 = 100
K = 105
r = 0.05
sigma = 0.2
q = 0.01
T = 1
mu = 0.08
M = 1000
N = 252
pct = [5, 50, 95] # percentiles of gain/loss for the hedged portfolio

results = simulated_delta_hedge_profit(S0, K, r, sigma, q, T, mu, M, N, pct)
for i in pct:
    print(i, results[i])

5 -2.408400583704496
50 -1.0039211267058707
95 -0.006920778315411293


In [ ]:
# Exercise 7.12
# Using Merton's formula to determine option price
def merton_call_option(S, K, P, sigma_s, sigma_r, rho, q, T):
    """
    S = current stock price.
    K = strike price of the option.
    P = bond price with maturity T (discount factor).
    sigma_s = volatility of the stock.
    sigma_r = volatility of the bond (interest rate volatility).
    rho = correlation between the stock and bond.
    q = dividend yield of the stock.
    T = time to maturity.
    """
    sigma_F = np.sqrt(sigma_s**2 + (sigma_r*T)**2 - 2 * rho * sigma_s * sigma_r*T)

    r = np.log(P)

    d1 = (np.log(S/K) + (r - q + 0.5*sigma**2)*T) / (sigma_F * np.sqrt(T))
    d2 = d1 - sigma_F*np.sqrt(T)

    call = np.exp(-q*T) * S * norm.cdf(d1) - np.exp(-r*T) * K * norm.cdf(d2)

    return call

In [12]:
# Set parameters
S = 100
K = 105
P = 0.95
sigma_s = 0.2
sigma_r = 0.1
rho = 0.5
q = 0.02
T = 1

call = merton_call_option(S, K, P, sigma_s, sigma_r, rho, q, T)
print(call)

2.5973143486870285
